# BigQuery Quota Exploration

Use this notebook to dry-run queries before executing them.
Always check the GB estimate before running anything against the full dataset.

In [ ]:
import sys
sys.path.insert(0, '..')

from pipeline.utils.bq_client import dry_run_bytes, get_client
import pipeline.config as cfg

In [ ]:
# ── How big is the github_events table for a single year? ──────────────────
q = """
SELECT COUNT(*) as n, SUM(1) as rows
FROM `bigquery-public-data.github_events`
WHERE type = 'PushEvent'
  AND _PARTITIONTIME BETWEEN '2023-01-01' AND '2023-12-31'
"""
gb = dry_run_bytes(q) / 1e9
print(f"Scan for PushEvents in 2023: {gb:.1f} GB")

In [ ]:
# ── TABLESAMPLE 1% of 2015-2018 — what does it cost? ─────────────────────
from pipeline import config as cfg

q2 = f"""
SELECT DISTINCT actor.login
FROM `{cfg.BQ_SOURCE_TABLE}`
TABLESAMPLE SYSTEM (1 PERCENT)
WHERE type = 'PushEvent'
  AND _PARTITIONTIME BETWEEN '2015-01-01' AND '2018-12-31'
LIMIT 100
"""
gb2 = dry_run_bytes(q2) / 1e9
print(f"Sampling query (1%) estimate: {gb2:.1f} GB")

In [ ]:
# ── Explore a handful of users manually ──────────────────────────────────
# Replace with real logins after running step 01 + 03

test_logins = ['torvalds', 'gvanrossum', 'dhh', 'antirez']

from pipeline.utils.gh_client import get_user
for login in test_logins:
    profile = get_user(login)
    if profile:
        print(f"{login:20s}  joined: {profile['created_at'][:10]}  repos: {profile['public_repos']}")

In [ ]:
# ── Check yearly commits for a test user via GraphQL ─────────────────────
from pipeline.utils.gh_client import get_contributions_by_year

login = 'torvalds'
years = list(range(2015, 2026))
contribs = get_contributions_by_year(login, years)

print(f"\n{login} contributions by year:")
for y, n in sorted(contribs.items()):
    bar = '█' * min(int(n / 20), 60)
    print(f"  {y}: {n:>5}  {bar}")